In [2]:

import sys
from pathlib import Path

PROJECT_ROOT = Path(rf"D:\Duan\agent_project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag_core import (
    RAGConfig,
    build_rag_agent,
    extract_evidence,
    validate_citations,
    run_agent
)

# 初始化整个 RAG Agent
config = RAGConfig.from_env()

bundle = build_rag_agent(config)

print("初始化完成")
print("Qdrant 文档数量:", len(bundle.documents))
print("Embedding:", bundle.document_embeddings.shape)
print("工具:", [t.name for t in bundle.tools])


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.462 seconds.
Prefix dict has been built successfully.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

初始化完成
Qdrant 文档数量: 7
Embedding: (7, 512)
工具: ['search_knowledge_base', 'calculate_effective_rainfall', 'calculate_rainfall_statistics']


In [8]:
query = "地下水位升高为什么会影响边坡稳定性？"

result = run_agent(bundle, query)

answer = result["messages"][-1].content

print(answer)

## 回答

根据知识库资料，地下水位升高影响边坡稳定性，主要通过以下机制：

1. **孔隙水压力升高**：地下水位升高时，坡体内部孔隙水压力可能增加 [S4]。
2. **有效应力减小**：孔隙水压力升高会减小有效应力 [S4]；与降雨入渗引起孔隙水压力升高的情况类似，有效应力降低的同时也会降低土体的抗剪强度 [S2]。
3. **边坡更易失稳**：有效应力减小使边坡更容易发生失稳 [S4]。

即总体链条为：地下水位升高，导致孔隙水压力增加，再导致有效应力（及抗剪强度）降低，最终使边坡更容易失稳 [S2][S4]。此外，资料明确指出地下水位变化会影响边坡稳定性 [S3]。

## 证据说明

- [S4]：直接说明地下水位升高会使坡体内部孔隙水压力可能增加，从而减小有效应力，使边坡更容易发生失稳。
- [S2]：说明降雨入渗使土体含水率增加、孔隙水压力升高，从而降低土体有效应力和抗剪强度（可与 S4 的机制相互印证，但针对的是降雨入渗而非地下水位升高）。
- [S3]：定性指出地下水位变化会影响边坡稳定性。
- [S6]：指出滑坡发生还受岩性、土体结构、植被覆盖和水文条件等因素共同控制，可用于说明边坡稳定性并非仅由地下水位决定。

## 资料局限性

- 知识库资料中未提供有效应力与孔隙水压力的定量关系式、抗剪强度公式（如莫尔–库仑准则中的具体参数）、安全系数的计算方法，也未给出任何数值阈值或案例实测数据。
- 关于地下水位升高是否额外引起坡体自重增加、渗流力（动水压力）增大等机制，现有资料未涉及，故不作推断。
- 资料仅有定性表述（如“可能增加”“更容易失稳”），不能据此推导定量结论。


In [9]:
query = "降雨量为120 mm，入渗量为35 mm，计算有效降雨量。"

result = run_agent(bundle, query)

print(result["messages"][-1].content)

## 回答

按简化水量平衡估算，有效降雨量为 **85 mm**，也就是总降雨量 120 mm 减去入渗量 35 mm 的结果（120 − 35 = 85 mm）。

## 证据说明

计算采用知识库工具中的简化有效降雨量公式，“总降雨量减入渗量”；本次输入的总降雨量为 120 mm，入渗量为 35 mm，因此得到 85 mm。该工具还给出径流系数估算值约 0.71。

## 资料局限性

该结果只是简化水量平衡估算，不等同于实测径流量；知识库资料未提供更精细的产流机制、下垫面参数或实测验证数据，因此不宜把它当作精确径流计算结果使用。


In [10]:

evaluation_questions = [
    {
        "id": 1,
        "query": "地下水位升高为什么会影响边坡稳定性？",
        "category": "landslide"
    },
    {
        "id": 2,
        "query": "持续降雨如何影响土体孔隙水压力？",
        "category": "landslide"
    },
    {
        "id": 3,
        "query": "坡度对滑坡发生有什么影响？",
        "category": "landslide"
    },
    {
        "id": 4,
        "query": "肯塔基州连续降雨达到多少毫米一定会发生滑坡？",
        "category": "unknown"
    },
    {
        "id": 5,
        "query": "太阳辐射为什么会影响光伏组件的发电效率？",
        "category": "photovoltaic"
    },
    {
        "id": 6,
        "query": "WRF 4.5.1 是什么？",
        "category": "meteorology"
    },
    {
        "id": 7,
        "query": "请计算有效降雨量，总降雨量120mm，入渗量35mm。",
        "category": "calculation"
    },
    {
        "id": 8,
        "query": "2026年某光伏电站的实际发电量是多少？",
        "category": "unknown"
    }
]

print("测试问题数量:", len(evaluation_questions))


测试问题数量: 8


In [5]:

import pandas as pd
import time

evaluation_results = []

for item in evaluation_questions:

    print("=" * 70)
    print("测试问题:", item["query"])

    start_time = time.time()

    try:
        result = run_agent(
            bundle,
            item["query"]
        )

        answer = result["messages"][-1].content

        # 提取本轮检索证据
        evidence_map = extract_evidence(
            result["messages"]
        )

        # 引用校验
        citation_report = validate_citations(
            answer,
            evidence_map
        )

        elapsed = time.time() - start_time

        evaluation_results.append({
            "id": item["id"],
            "query": item["query"],
            "category": item["category"],
            "answer": answer,
            "evidence_count": len(evidence_map),
            "cited_ids": citation_report["cited_ids"],
            "invalid_citations": citation_report["invalid_citations"],
            "metadata_problems": citation_report["metadata_problems"],
            "citation_validity": citation_report["citation_validity"],
            "metadata_complete": citation_report["metadata_complete"],
            "latency_s": elapsed
        })

        print(answer)
        print("证据数量:", len(evidence_map))
        print("引用校验:", citation_report)

    except Exception as e:
        print("测试失败:", str(e))

        evaluation_results.append({
            "id": item["id"],
            "query": item["query"],
            "category": item["category"],
            "answer": None,
            "error": str(e)
        })

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)


测试问题: 地下水位升高为什么会影响边坡稳定性？
## 回答

地下水位升高主要会通过**孔隙水压力**这一中间环节削弱边坡稳定性：地下水位升高时，坡体内部孔隙水压力可能增加，这会减小有效应力，使边坡更容易发生失稳 [S4]。

与此对应，降雨入渗会使土体含水率增加，并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度 [S2]。因此，从力学角度看，地下水位上升使有效应力下降，进而降低土体的抗剪强度，从而不利于边坡稳定 [S2][S4]。

此外，资料也指出地下水位变化会影响边坡稳定性 [S3]，并且滑坡是否发生还受岩性、土体结构、植被覆盖和水文条件等多种因素共同控制 [S6]，并非只由地下水位单一因素决定。

## 证据说明

- [S4]：直接说明地下水位升高会使坡体内部孔隙水压力增加，进而减小有效应力，使边坡更易失稳。
- [S2]：说明降雨入渗使含水率增加、孔隙水压力升高，降低有效应力与抗剪强度，可作为“有效应力降低至抗剪强度降低”这一过程链的支持证据。
- [S3]：确认地下水位变化是影响边坡稳定性的因素之一。
- [S6]：说明滑坡发生受岩性、土体结构、植被覆盖和水文条件等多种因素共同控制，用于限定单一因素的解释边界。
- [S1]、[S5]：分别涉及持续降雨诱发滑坡、坡度作为地形条件，与本题因果关系无直接关联，未作为支持证据使用。

## 资料局限性

现有资料仅给出定性表述，即“孔隙水压力增加→有效应力减小→更易失稳”，未提供：

- 有效应力与抗剪强度的具体计算式或参数（如内摩擦角、黏聚力取值）；
- 地下水位升幅与稳定系数变化之间的定量关系或临界水位；
- 具体的试验数据、监测数值或案例结果。

因此，上述机制无法给出定量结论。
证据数量: 6
引用校验: {'cited_ids': ['S4', 'S2', 'S3', 'S6', 'S1', 'S5'], 'valid_citations': ['S4', 'S2', 'S3', 'S6', 'S1', 'S5'], 'invalid_citations': [], 'metadata_problems': [], 'citation_validity': True, 'metadata_complete': True}
测试问题: 持续降雨如何影响土体孔隙水压力？
## 回答

根据知识

,id,query,category,answer,evidence_count,cited_ids,invalid_citations,metadata_problems,citation_validity,metadata_complete,latency_s
0,1,地下水位升高为什么会影响边坡稳定性？,landslide,## 回答\n\n地下水位升高主要会通过**孔隙水压力**这一中间环节削弱边坡稳定性：地下水...,6,"[S4, S2, S3, S6, S1, S5]",[],[],True,True,5.314949
1,2,持续降雨如何影响土体孔隙水压力？,landslide,## 回答\n\n根据知识库资料，持续降雨影响土体孔隙水压力的主要定性机制是：降雨入渗使土体...,6,"[S2, S4, S1]",[],[],True,True,4.788961
2,3,坡度对滑坡发生有什么影响？,landslide,## 回答\n\n根据知识库资料，坡度是影响滑坡发生的重要地形条件 [S5]。较大的坡度通常...,6,"[S5, S6, S4, S2]",[],[],True,True,3.421510
3,4,肯塔基州连续降雨达到多少毫米一定会发生滑坡？,unknown,## 回答\n\n根据当前知识库资料，无法确定“肯塔基州连续降雨达到多少毫米一定会发生滑坡”...,6,"[S1, S2, S4, S6, S5]",[],[],True,True,4.805021
4,5,太阳辐射为什么会影响光伏组件的发电效率？,photovoltaic,## 回答\n\n根据当前知识库资料，只能确定太阳辐射是光伏功率预测中最重要的气象因素之一，...,7,"[S7, S1, S6]",[],[],True,True,3.832976
5,6,WRF 4.5.1 是什么？,meteorology,## 回答\n\n根据当前知识库资料，无法确定该问题。\n\n检索到的资料中没有关于 “WR...,5,"[S2, S4, S5, S6, S7]",[],[],True,True,3.453537
6,7,请计算有效降雨量，总降雨量120mm，入渗量35mm。,calculation,## 回答\n\n有效降雨量 = 总降雨量 − 入渗量 = 120 mm − 35 mm =...,4,"[S2, S1, S3, S4]",[],[],True,True,4.930120
7,8,2026年某光伏电站的实际发电量是多少？,unknown,## 回答\n\n根据当前知识库资料，无法确定该问题。\n\n知识库中没有任何关于“2026...,5,"[S7, S2, S4, S6]",[],[],True,True,3.125019


In [11]:

valid_df = evaluation_df[
    evaluation_df["citation_validity"].notna()
]

summary = {
    "测试问题总数": len(evaluation_df),
    "成功执行数量": len(valid_df),
    "平均检索证据数": valid_df["evidence_count"].mean(),
    "引用有效率": valid_df["citation_validity"].mean(),
    "来源页码完整率": valid_df["metadata_complete"].mean(),
    "平均响应时间(s)": valid_df["latency_s"].mean()
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["指标", "结果"]
)

display(summary_df)

# 保存结果
evaluation_df.to_excel(
    "agent_citation_evaluation.xlsx",
    index=False
)


,指标,结果
0,测试问题总数,8.000000
1,成功执行数量,8.000000
2,平均检索证据数,5.625000
3,引用有效率,1.000000
4,来源页码完整率,1.000000
5,平均响应时间(s),4.209012


In [12]:
invalid_df = evaluation_df[
    evaluation_df["citation_validity"] == False
]

display(
    invalid_df[
        ["id", "query", "answer", "cited_ids",
         "invalid_citations"]
    ]
)

,id,query,answer,cited_ids,invalid_citations


In [13]:
reference_answers = {
    1: "地下水位升高时，坡体内部孔隙水压力可能增加，从而减小有效应力，使边坡更容易发生失稳。",

    2: "持续降雨入渗会增加土体含水率，并可能提高孔隙水压力，从而降低土体有效应力和抗剪强度。",

    3: "坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，但滑坡还受岩性、土体结构、植被覆盖和水文条件等因素控制。",

    5: "太阳辐射是影响光伏发电功率的重要气象因素，云量、温度、湿度和风速也会影响光伏组件的发电效率。",

    6: "WRF 4.5.1 是一种数值天气预报模型版本。"
}

In [14]:
ragas_dataset = []

for item in evaluation_questions:

    if item["id"] not in reference_answers:
        continue

    result = run_agent(bundle, item["query"])

    answer = result["messages"][-1].content

    evidence_map = extract_evidence(
        result["messages"]
    )

    contexts = [
        evidence["text"]
        for evidence in evidence_map.values()
    ]

    ragas_dataset.append({
        "user_input": item["query"],
        "response": answer,
        "retrieved_contexts": contexts,
        "reference": reference_answers[item["id"]]
    })

print("RAGAS 样本数量:", len(ragas_dataset))

RAGAS 样本数量: 5


In [15]:
import os
import instructor

from openai import AsyncOpenAI
from ragas.llms.base import InstructorLLM

# 1. 原生异步 DeepSeek 客户端
deepseek_eval_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    timeout=60.0,
    max_retries=2
)

# 2. 用 Instructor 包装客户端
# MD_JSON 在此配置，而不是交给 llm_factory
instructor_client = instructor.from_openai(
    deepseek_eval_client,
    mode=instructor.Mode.MD_JSON
)

# 3. 直接创建 Ragas 0.4.3 评估模型
evaluator_llm = InstructorLLM(
    client=instructor_client,
    model="deepseek-chat",
    provider="openai",
    max_tokens=4096
)

print(evaluator_llm)
print("是否为异步模型:", evaluator_llm.is_async)
print("模型参数:", evaluator_llm.model_args)

InstructorLLM(provider='openai', model='deepseek-chat', client=<AsyncInstructor:async>, temperature=0.01, max_tokens=4096, top_p=0.1)
是否为异步模型: True
模型参数: {'temperature': 0.01, 'top_p': 0.1, 'max_tokens': 4096}


In [16]:
from ragas.embeddings import HuggingFaceEmbeddings

evaluator_embeddings = HuggingFaceEmbeddings(
    model="BAAI/bge-small-zh-v1.5",
    device="cpu",
    normalize_embeddings=True
)

print(evaluator_embeddings)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

HuggingFaceEmbeddings(provider='huggingface', model='BAAI/bge-small-zh-v1.5', client=<SentenceTransformer>, use_api=False, device='cpu')


In [17]:
# 初始化四个评价指标
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall
)

faithfulness_metric = Faithfulness(
    llm=evaluator_llm
)

answer_relevancy_metric = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

context_precision_metric = ContextPrecisionWithReference(
    llm=evaluator_llm
)

context_recall_metric = ContextRecall(
    llm=evaluator_llm
)

In [18]:
ragas_results = []

for item in ragas_dataset:

    # 1. 忠实度
    f = await faithfulness_metric.ascore(
        user_input=item["user_input"],
        retrieved_contexts=item["retrieved_contexts"],
        response=item["response"]
    )

    # 2. 回答相关性
    ar = await answer_relevancy_metric.ascore(
        user_input=item["user_input"],
        response=item["response"]
    )

    # 3. 上下文精确率
    cp = await context_precision_metric.ascore(
        user_input=item["user_input"],
        retrieved_contexts=item["retrieved_contexts"],
        reference=item["reference"]
    )

    # 4. 上下文召回率
    cr = await context_recall_metric.ascore(
        user_input=item["user_input"],
        retrieved_contexts=item["retrieved_contexts"],
        reference=item["reference"]
    )

    ragas_results.append({
        "question": item["user_input"],
        "Faithfulness": f.value,
        "AnswerRelevancy": ar.value,
        "ContextPrecision": cp.value,
        "ContextRecall": cr.value
    })

ragas_df = pd.DataFrame(ragas_results)

display(ragas_df.round(4))

,question,Faithfulness,AnswerRelevancy,ContextPrecision,ContextRecall
0,地下水位升高为什么会影响边坡稳定性？,1.0000,0.9704,0.8333,1.0
1,持续降雨如何影响土体孔隙水压力？,0.8125,0.9133,1.0000,1.0
2,坡度对滑坡发生有什么影响？,0.4545,0.9611,1.0000,1.0
3,太阳辐射为什么会影响光伏组件的发电效率？,0.8571,0.0000,1.0000,1.0
4,WRF 4.5.1 是什么？,0.6471,0.0000,0.0000,0.0


In [20]:
# =====================================================
# 检查 RAGAS 低分样本
# =====================================================
# prompt = RAGConfig.system_prompt
# print(prompt)
for i, item in enumerate(ragas_dataset):

    print("\n" + "=" * 80)
    print(f"问题 {i + 1}")
    print("=" * 80)

    print("\n【用户问题】")
    print(item["user_input"])

    print("\n【Agent 实际回答】")
    print(item["response"])

    print("\n【检索上下文】")

    for j, context in enumerate(
        item["retrieved_contexts"], 1
    ):
        print(f"\nContext {j}:")
        print(context)

    print("\n【参考答案】")
    print(item["reference"])

    print("\n【RAGAS 评分】")
    print(ragas_df.iloc[i][[
        "Faithfulness",
        "AnswerRelevancy",
        "ContextPrecision",
        "ContextRecall"
    ]])

    print("\n" + "-" * 80)


问题 1

【用户问题】
地下水位升高为什么会影响边坡稳定性？

【Agent 实际回答】
## 回答

根据知识库资料，地下水位升高影响边坡稳定性的核心机制是：地下水位升高会使**坡体内部孔隙水压力增加，从而减小有效应力，使边坡更容易发生失稳** [S4]。资料同时指出，地下水位变化会影响边坡稳定性 [S3]。

与该机制相印证的证据表明：降雨入渗会导致土体含水率增加，并可能造成孔隙水压力升高，从而**降低土体有效应力和抗剪强度** [S2]。也就是说，孔隙水压力升高通过降低有效应力来削弱土体的抗剪强度，这是边坡稳定性下降的原因。

另外，滑坡的发生并不只取决于水文条件，还受到岩性、土体结构、植被覆盖和坡度等地形条件的共同控制 [S6]。

## 证据说明

- [S4]（landslide_report.pdf，第12页）：给出了直接机理——地下水位升高 → 孔隙水压力增加 → 有效应力减小 → 边坡更易失稳。
- [S3]（同文件第12页）：说明地下水位变化会影响边坡稳定性。
- [S2]（同文件第12页）：说明孔隙水压力升高会降低有效应力与抗剪强度。
- [S6]（同文件第12页）：说明滑坡还受岩性、土体结构、植被覆盖和水文条件共同控制。

## 资料局限性

知识库仅提供定性描述，**未提供**有效应力公式、抗剪强度参数（如黏聚力、内摩擦角）、安全系数计算方法、孔隙水压力与水位升高的定量关系，也未给出具体数值或案例计算结果。因此无法据此给出定量评估。

【检索上下文】

Context 1:
当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

Context 2:
地下水位变化同样会影响边坡稳定性。

Context 3:
降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

Context 4:
坡度是影响滑坡发生的重要地形条件。

Context 5:
较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

Context 6:
持续降雨是诱发滑坡的重要因素之一。

【参考答案】
地下水位升高时，坡体内部孔隙水压力可能增加，从而减小有效应力，使边坡更容易发生失稳。

【RAGAS 评分】
Faithfulness    

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler()
    ],
    force=True
)

print("日志系统初始化完成")

日志系统初始化完成


In [2]:
import importlib
import rag_core

importlib.reload(rag_core)

# 重新导入
from rag_core import (
    RAGConfig,
    build_rag_agent,
    run_agent,
    extract_evidence,
    validate_citations
)

config = RAGConfig.from_env()

bundle = build_rag_agent(config)

print("Agent 初始化完成")

2026-11-16 22:28:16,158 | INFO | httpx | HTTP Request: GET http://localhost:6333/collections/rag_knowledge "HTTP/1.1 200 OK"
2026-11-16 22:28:16,171 | INFO | httpx | HTTP Request: POST http://localhost:6333/collections/rag_knowledge/points/scroll "HTTP/1.1 200 OK"
2026-11-16 22:28:16,207 | INFO | sentence_transformers.base.model | No device provided, using cuda:0
2026-11-16 22:28:16,208 | INFO | httpx | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-11-16 22:28:16,843 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5 "HTTP/1.1 200 OK"
2026-11-16 22:28:16,848 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-small-zh-v1.5.


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

2026-11-16 22:28:17,393 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-16 22:28:17,707 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-16 22:28:18,044 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-zh-v1.5/resolve/7999e1d3359715c523056ef9478215996d62a620/2_Normalize/config.json "HTTP/1.1 404 Not Found"
2026-11-16 22:28:18,045 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...
2026-11-16 22:28:18,391 | DEBUG | jieba | Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
2026-11-16 22:28:18,393 | DEBUG | jieba | Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.478 seconds.
2026-11-16 22:28:18,871 | DEBUG | jieba | Loading model cost 0.478 seconds.
Prefix dict has been built successfully.
2026-11-16 22:28:18,872 | DEBUG | jieba | Prefix dict has been built successfully.
2026-11-16 22:28:18,878 | INFO | sentence_transformers.base.model | No device provided, using cuda:0
2026-11-16 22:28:19,198 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3 "HTTP/1.1 200 OK"
2026-11-16 22:28:19,523 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/modules.json "HTTP/1.1 404 Not Fou

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

2026-11-16 22:28:21,954 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-16 22:28:22,283 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-16 22:28:24,096 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3 "HTTP/1.1 200 OK"


Agent 初始化完成


In [3]:
thread_id = "test_session_001"

result1 = run_agent(
    bundle,
    "地下水位升高为什么会影响边坡稳定性？",
    thread_id=thread_id
)

print(result1["messages"][-1].content)

2026-11-16 22:29:02,403 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-16 22:29:04,045 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-11-16 22:29:06,491 | INFO | RAGAgent | Agent运行成功 | thread_id=test_session_001 | latency=4.955s


## 回答

根据知识库资料，地下水位升高影响边坡稳定性的主要机理是：**地下水位升高会增大坡体内部孔隙水压力，从而减小有效应力，使边坡更容易发生失稳** [S4]。

与这一机理相关的证据还指出：降雨入渗使土体含水率增加、孔隙水压力升高，会降低土体的有效应力和抗剪强度 [S2]。也就是说，地下水位升高通过“孔隙水压力升高 → 有效应力降低 → 抗剪强度下降”这一途径削弱边坡的稳定性。此外，资料也确认地下水位变化会影响边坡稳定性 [S3]，而滑坡是否发生还受岩性、土体结构、植被覆盖和水文条件等因素共同控制 [S6]。

## 证据说明

- [S4]（landslide_report.pdf, p.12）：明确指出地下水位升高 → 孔隙水压力可能增加 → 有效应力减小 → 边坡更易失稳。
- [S2]（landslide_report.pdf, p.12）：降雨入渗 → 含水率增加、孔隙水压力升高 → 降低有效应力与抗剪强度，可与 [S4] 相互印证。
- [S3]（landslide_report.pdf, p.12）：定性说明地下水位变化会影响边坡稳定性。
- [S6]（landslide_report.pdf, p.12）：说明滑坡受多因素共同控制，提示单因素解释的局限性。

## 资料局限性

现有资料只给出定性描述，**未提供**安全系数计算公式、有效应力或抗剪强度的具体参数取值、孔隙水压力与水位升幅之间的定量关系，也未给出边坡稳定性系数随地下水位变化的具体数值结果。若需定量评估，当前知识库资料不足。


In [4]:
result2 = run_agent(
    bundle,
    "那它主要通过什么机制降低边坡稳定性？",
    thread_id=thread_id
)

print(result2["messages"][-1].content)

2026-11-16 22:29:32,520 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-16 22:29:34,541 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-11-16 22:29:37,501 | INFO | RAGAgent | Agent运行成功 | thread_id=test_session_001 | latency=5.132s


## 回答

根据当前知识库资料，地下水位升高主要通过**“孔隙水压力升高 → 有效应力减小 → 抗剪强度降低”**这一力学机制降低边坡稳定性：

1. **孔隙水压力升高**：地下水位升高时，坡体内部孔隙水压力可能增加 [S4]。
2. **有效应力减小**：孔隙水压力增加会减小土体有效应力 [S4]；在降雨入渗条件下，土体含水率增加同样会造成孔隙水压力升高，从而降低有效应力 [S2]。
3. **抗剪强度降低**：有效应力的降低伴随土体抗剪强度的降低 [S2]。
4. **边坡更易失稳**：有效应力减小后，边坡更容易发生失稳 [S4]。

需要说明的是，知识库将地下水位变化视为影响边坡稳定性的一个因素 [S3]，但滑坡发生还受岩性、土体结构、植被覆盖和水文条件等多因素共同控制 [S6]，因此该机制并非唯一控制因素。

## 证据说明

- [S4]：直接给出“地下水位升高 → 孔隙水压力增加 → 有效应力减小 → 更易失稳”的因果链。
- [S2]：补充“含水率增加 → 孔隙水压力升高 → 有效应力与抗剪强度降低”，与 [S4] 相互印证。
- [S3]、[S6]：说明地下水位变化影响边坡稳定性，且滑坡受多因素共同控制。

## 资料局限性

知识库仅提供上述定性机制，**未涉及**渗透力/渗流力、静水压力与浮托力、土体软化或结构面润滑等其他可能机制的说明，也未给出有效应力公式、抗剪强度参数（黏聚力、内摩擦角）及孔隙水压力与水位升幅的定量关系。若需更完整的机制清单或定量评价，当前资料不足。


In [5]:
result3 = run_agent(
    bundle,
    "那它主要通过什么机制降低边坡稳定性？",
    thread_id="test_session_002"
)

print(result3["messages"][-1].content)

2026-11-16 22:31:38,170 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-16 22:31:39,590 | INFO | httpx | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-11-16 22:31:42,275 | INFO | RAGAgent | Agent运行成功 | thread_id=test_session_002 | latency=4.233s


## 回答

根据知识库资料，这一因素主要借助**水文—力学途径**降低边坡稳定性。按“降雨/地下水位变化”来理解，其作用机制为：降雨入渗使土体含水率增加，并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度 [S2]；当地下水位升高时，坡体内部孔隙水压力可能增加，同样会减小有效应力，使边坡更容易发生失稳 [S4]。持续降雨则是诱发滑坡的重要外部因素 [S1]。

## 证据说明

- [S2]：降雨入渗 → 含水率增加 → 孔隙水压力可能升高 → 有效应力与抗剪强度降低。
- [S4]：地下水位升高 → 孔隙水压力增加 → 有效应力减小 → 失稳可能性增大。
- [S1]：持续降雨是诱发滑坡的重要因素之一。
- [S3]：仅定性指出地下水位变化会影响边坡稳定性，未给出具体机制细节。
- [S5][S6]：涉及坡度等地形条件，属于控制滑坡的并列因素，而非本次所问的降低稳定性机制。

## 资料局限性

知识库仅给出上述定性因果链，未提供有效应力公式、抗剪强度参数（如 c、φ）、孔隙水压力数值、入渗量或稳定性系数的定量结果；也未说明各机制的相对贡献大小。若“它”指代其他具体因素（如坡度、植被等），当前资料不足以确定对应机制，请明确所指对象。


In [6]:
history = bundle.graph.get_state({
    "configurable": {
        "thread_id": "test_session_001"
    }
})

print("历史消息数量：", len(history.values["messages"]))

for msg in history.values["messages"]:
    print(
        type(msg).__name__,
        str(msg.content)[:200]
    )

历史消息数量： 10
HumanMessage 地下水位升高为什么会影响边坡稳定性？
AIMessage 
ToolMessage [
  {
    "evidence_id": "S4",
    "rank": 1,
    "id": 3,
    "point_id": 3,
    "document_id": "doc_001",
    "chunk_id": 3,
    "text": "当地下水位升高时，\n坡体内部孔隙水压力可能增加，从而减小有效应力，\n使边坡更容易发生失稳。",
    "sourc
ToolMessage [
  {
    "evidence_id": "S2",
    "rank": 1,
    "id": 1,
    "point_id": 1,
    "document_id": "doc_001",
    "chunk_id": 1,
    "text": "降雨入渗会导致土体含水率增加，\n并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。",
    "source
AIMessage ## 回答

根据知识库资料，地下水位升高影响边坡稳定性的主要机理是：**地下水位升高会增大坡体内部孔隙水压力，从而减小有效应力，使边坡更容易发生失稳** [S4]。

与这一机理相关的证据还指出：降雨入渗使土体含水率增加、孔隙水压力升高，会降低土体的有效应力和抗剪强度 [S2]。也就是说，地下水位升高通过“孔隙水压力升高 → 有效应力降低 → 抗剪强度下降”这一途径削弱边坡的稳定性。此外，资料也
HumanMessage 那它主要通过什么机制降低边坡稳定性？
AIMessage 
ToolMessage [
  {
    "evidence_id": "S4",
    "rank": 1,
    "id": 3,
    "point_id": 3,
    "document_id": "doc_001",
    "chunk_id": 3,
    "text": "当地下水位升高时，\n坡体内部孔隙水压力可能增加，从而减小有效应力，\n使边坡更容易发生失稳。",
    "sourc
ToolMessage [
  {
    "evidence_id": "S2",
    "rank"

In [7]:
import importlib
import rag_core

importlib.reload(rag_core)

from rag_core import RAGConfig, build_rag_agent

config = RAGConfig.from_env()

bundle = build_rag_agent(config)

2026-11-17 15:41:24,016 | INFO | httpx | HTTP Request: GET http://localhost:6333/collections/rag_knowledge "HTTP/1.1 200 OK"
2026-11-17 15:41:24,025 | INFO | httpx | HTTP Request: POST http://localhost:6333/collections/rag_knowledge/points/scroll "HTTP/1.1 200 OK"
2026-11-17 15:41:24,032 | INFO | sentence_transformers.base.model | No device provided, using cuda:0
2026-11-17 15:41:24,039 | INFO | httpx | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-11-17 15:41:25,788 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5 "HTTP/1.1 200 OK"
2026-11-17 15:41:25,799 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-small-zh-v1.5.


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

2026-11-17 15:41:26,271 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-17 15:41:26,581 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-zh-v1.5/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-17 15:41:26,929 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-zh-v1.5/resolve/7999e1d3359715c523056ef9478215996d62a620/2_Normalize/config.json "HTTP/1.1 404 Not Found"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-17 15:41:26,990 | INFO | sentence_transformers.base.model | No device provided, using cuda:0
2026-11-17 15:41:27,316 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3 "HTTP/1.1 200 OK"
2026-11-17 15:41:27,628 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/modules.json "HTTP/1.1 404 Not Found"
2026-11-17 15:41:27,637 | INFO | sentence_transformers.base.model | No modules.json found for BAAI/bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

2026-11-17 15:41:30,763 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-17 15:41:31,066 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-17 15:41:33,337 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3 "HTTP/1.1 200 OK"


In [8]:
import time
import pandas as pd

query = "地下水位升高为什么会影响边坡稳定性？"

# 清空缓存
bundle.clear_retrieval_cache()

results = []

# 第一次：冷启动
start = time.perf_counter()

result1 = bundle.hybrid_rerank_search(
    query,
    top_k=4
)

cold_time = time.perf_counter() - start

results.append({
    "测试": "第一次检索（无缓存）",
    "耗时(s)": cold_time
})

# 第二次：命中缓存
start = time.perf_counter()

result2 = bundle.hybrid_rerank_search(
    query,
    top_k=4
)

warm_time = time.perf_counter() - start

results.append({
    "测试": "第二次检索（有缓存）",
    "耗时(s)": warm_time
})

# 检查结果一致性
print("检索结果是否一致：", result1 == result2)

# 输出性能对比
performance_df = pd.DataFrame(results)

display(performance_df.round(4))

# 缓存命中情况
print("\n缓存状态：")

display(pd.DataFrame(
    bundle.get_cache_stats()
))

检索缓存已清理


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

检索结果是否一致： True


,测试,耗时(s)
0,第一次检索（无缓存）,0.0613
1,第二次检索（有缓存）,0.0001



缓存状态：


,embedding_cache,rerank_cache
hits,0,1
misses,1,1
maxsize,1024,512
currsize,1,1


In [9]:
test_queries = [
    "地下水位升高为什么会影响边坡稳定性？",
    "持续降雨如何影响土体孔隙水压力？",
    "坡度对滑坡发生有什么影响？",
    "地下水位升高为什么会影响边坡稳定性？",
    "持续降雨如何影响土体孔隙水压力？"
]

performance = []

for query in test_queries:

    start = time.perf_counter()

    results = bundle.hybrid_rerank_search(
        query,
        top_k=4
    )

    elapsed = time.perf_counter() - start

    performance.append({
        "query": query,
        "结果数量": len(results),
        "耗时(s)": elapsed
    })

performance_df = pd.DataFrame(performance)

display(performance_df.round(4))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,query,结果数量,耗时(s)
0,地下水位升高为什么会影响边坡稳定性？,4,0.0001
1,持续降雨如何影响土体孔隙水压力？,4,0.0609
2,坡度对滑坡发生有什么影响？,4,0.0665
3,地下水位升高为什么会影响边坡稳定性？,4,0.0000
4,持续降雨如何影响土体孔隙水压力？,4,0.0000
